# Notebook 2: Zero-Shot Baseline & QLoRA SFT Training (Kaggle)
**Project**: Autonomous Text-to-SQL Generator Aligned via RLVR + GRPO  
**Hardware**: Kaggle GPU (T4 x 1, P100, or 2xT4 — pinned to single GPU to avoid DataParallel + 4-bit conflicts)  
**Phase**: Phase 2 (Sandbox Verifier) & Phase 3 (Zero-Shot Baseline & Supervised Fine-Tuning)

---

In [ ]:
# Step 1: Install a pinned, compatible training stack for Kaggle GPU training
# - TRL 0.25.0 uses the standard causal-LM loss path (avoiding fused LM-head issues).
# - bitsandbytes < 0.50.0 avoids the 0.50+ gemm_4bit / cuBLASLt fallback regression on T4 (CC 7.5).
# - Pin Transformers 4.x and PEFT >= 0.18 for stable QLoRA compatibility.
!pip uninstall -y -q torchvision
!pip install -q \
    "transformers==4.57.3" \
    "trl==0.25.0" \
    "peft==0.18.1" \
    "datasets>=3.0.0,<5.0.0" \
    "accelerate>=1.2.1,<2.0.0" \
    "bitsandbytes>=0.45.2,<0.50.0" \
    "sqlglot>=26.0.0"
print("Pinned ML training packages installed. Restart the Kaggle session before running the remaining cells.")

In [ ]:
# Step 2: GPU Inspection & Path Setup (With Auto-Unpack for Kaggle Inputs)
import os
import sys

# CRITICAL: Must be set BEFORE importing torch!
# 1. Pin to GPU 0 so SFTTrainer never wraps 4-bit QLoRA in legacy nn.DataParallel on Kaggle T4 x2
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# 2. Reduce CUDA allocator fragmentation on 15 GB T4
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Fully block torchvision to prevent Kaggle CUDA/PyTorch operator ABI mismatch
sys.modules["torchvision"] = None
sys.modules["torchvision.transforms"] = None
sys.modules["torchvision.ops"] = None

import shutil
import zipfile
from pathlib import Path
import torch

assert torch.cuda.is_available(), "CUDA GPU is required! Please enable GPU in Kaggle Notebook Settings."
major, minor = torch.cuda.get_device_capability(0)
print(f"Visible CUDA Devices: {torch.cuda.device_count()} (pinned to single GPU for QLoRA stability)")
print(f"CUDA Device: {torch.cuda.get_device_name(0)} (Compute Capability {major}.{minor})")
print(f"Allocated Memory: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
print(f"Hardware BF16 Supported (requires CC >= 8.0): {major >= 8}")

WORKING_DIR = Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
OUTPUTS_DIR = WORKING_DIR / "outputs" / "sft"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Auto-unpack data from uploaded dataset (sql_grpo_data.zip, sql_grpo_processed_data.zip, etc.)
train_manifest = DATA_DIR / "processed" / "train_manifest.jsonl"
dev_manifest = DATA_DIR / "processed" / "dev_manifest.jsonl"

def setup_dataset():
    if train_manifest.exists() and dev_manifest.exists():
        return

    # 1. Search for any zip archive containing train_manifest.jsonl or matching data naming
    all_zips = list(Path("/kaggle/input").glob("**/*.zip")) + list(WORKING_DIR.glob("*.zip"))
    for z in all_zips:
        if any(k in z.name.lower() for k in ["sql_grpo", "data", "processed", "spider"]):
            try:
                with zipfile.ZipFile(z, "r") as zf:
                    names = zf.namelist()
                    if any("train_manifest.jsonl" in n for n in names):
                        print(f"📦 Unpacking dataset from archive: {z} ...")
                        has_data_root = any(n.startswith("data/") for n in names)
                        extract_target = WORKING_DIR if has_data_root else DATA_DIR
                        zf.extractall(extract_target)
                        print(f"Extracted into: {extract_target}")
                        break
            except Exception as e:
                print(f"Warning inspecting {z}: {e}")

    # 2. If archive was extracted to /kaggle/working/processed/ instead of /kaggle/working/data/processed/
    if not train_manifest.exists() and (WORKING_DIR / "processed" / "train_manifest.jsonl").exists():
        print(f"Relocating data folders to {DATA_DIR} ...")
        for folder in ["processed", "raw"]:
            src = WORKING_DIR / folder
            dst = DATA_DIR / folder
            if src.exists() and not dst.exists():
                shutil.move(src, dst)

    # 3. If Kaggle pre-extracted the dataset in /kaggle/input/ (without a zip file)
    if not train_manifest.exists():
        found_manifests = list(Path("/kaggle/input").glob("**/train_manifest.jsonl"))
        if found_manifests:
            m_path = found_manifests[0]
            dataset_root = m_path.parent.parent if m_path.parent.name == "processed" else m_path.parent
            print(f"📂 Found pre-extracted dataset at {dataset_root}. Syncing to {DATA_DIR} ...")
            src_proc = dataset_root / "processed" if (dataset_root / "processed").exists() else m_path.parent
            dst_proc = DATA_DIR / "processed"
            if src_proc.exists() and not dst_proc.exists():
                shutil.copytree(src_proc, dst_proc)
            src_raw = dataset_root / "raw" if (dataset_root / "raw").exists() else (dataset_root / "data" / "raw")
            dst_raw = DATA_DIR / "raw"
            if src_raw.exists() and not dst_raw.exists():
                shutil.copytree(src_raw, dst_raw)

setup_dataset()

if not (train_manifest.exists() and dev_manifest.exists()):
    available_inputs = [str(p) for p in Path("/kaggle/input").glob("*")]
    raise FileNotFoundError(
        "❌ CRITICAL ERROR: train_manifest.jsonl was not found in /kaggle/input or /kaggle/working!\n\n" +
        f"Items currently in /kaggle/input: {available_inputs}\n\n" +
        "👉 HOW TO FIX IN KAGGLE:\n" +
        "1. In the right-hand panel, under 'Input', click '+ Add Input'.\n" +
        "2. Select your uploaded 'sql_grpo_data' dataset (or upload 'sql_grpo_data.zip').\n" +
        "3. Re-run this cell."
    )

print(f"✅ Verified processed dataset at: {DATA_DIR}")
with open(train_manifest, "r", encoding="utf-8") as f:
    num_train = sum(1 for _ in f)
with open(dev_manifest, "r", encoding="utf-8") as f:
    num_dev = sum(1 for _ in f)
print(f"  - Train examples: {num_train}")
print(f"  - Dev examples:   {num_dev}")

In [ ]:
# Step 3: Hardened SQLite Sandbox Implementation
import sqlite3
import time
from collections import Counter
from typing import Any, List, Optional, Tuple

SAFE_FUNCTIONS = {"count", "sum", "avg", "min", "max", "lower", "upper", "length", "substr", "round", "abs", "coalesce", "strftime", "date"}

def create_authorizer():
    def callback(action, arg1, arg2, db, trigger):
        if action == sqlite3.SQLITE_SELECT:
            return sqlite3.SQLITE_OK
        if action == sqlite3.SQLITE_READ:
            if (arg1 or "").lower() in {"sqlite_master", "sqlite_schema", "sqlite_temp_master"}:
                return sqlite3.SQLITE_DENY
            return sqlite3.SQLITE_OK
        if action == sqlite3.SQLITE_FUNCTION:
            if (arg2 or "").lower() in SAFE_FUNCTIONS:
                return sqlite3.SQLITE_OK
            return sqlite3.SQLITE_DENY
        return sqlite3.SQLITE_DENY
    return callback

def execute_sandbox_query(db_path: Path, sql: str, timeout_sec: float = 2.0):
    start = time.perf_counter()
    deadline = start + timeout_sec
    uri = f"file:{db_path.resolve().as_posix()}?mode=ro"
    conn = None
    try:
        conn = sqlite3.connect(uri, uri=True, timeout=1.0)
        conn.execute("PRAGMA query_only = ON")
        conn.set_authorizer(create_authorizer())
        conn.set_progress_handler(lambda: 1 if time.perf_counter() > deadline else 0, 10000)
        cur = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchmany(5001)
        if len(rows) > 5000:
            return False, None, True, "RESULT_TRUNCATED"
        return True, rows, False, None
    except Exception as e:
        return False, None, False, str(e)
    finally:
        if conn:
            conn.close()

print("Sandbox and Authorizer ready.")

In [ ]:
# Step 4: Load Model & Tokenizer in 4-bit (Qwen2.5-Coder-1.5B-Instruct)
import sys
# Ensure torchvision cannot be imported by any internal transformers module
sys.modules["torchvision"] = None
sys.modules["torchvision.transforms"] = None
sys.modules["torchvision.ops"] = None

import transformers.utils.import_utils as _imp
_imp.is_torchvision_available = lambda: False

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# Detect GPU compute capability for optimal quantization (CC >= 8.0 required for native BF16 GEMM)
major, minor = torch.cuda.get_device_capability(0)
supports_bf16 = (major >= 8)
compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16
print(f"CUDA Capability: {major}.{minor} | Using 4-bit compute dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading tokenizer and 4-bit quantized model: {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Pin explicitly to cuda:0 (avoids multi-GPU sharding / DataParallel conflicts with 4-bit weights)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map={"": 0},
    trust_remote_code=True,
)
print(f"Model loaded successfully on cuda:0. Peak VRAM: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")

In [ ]:
# Step 5: Evaluate Zero-Shot Baseline (Pass@1 Greedy)
import json
import sqlglot

def extract_sql(text):
    if "<sql>" in text and "</sql>" in text:
        return text.split("<sql>")[1].split("</sql>")[0].strip()
    return ""

dev_manifest = DATA_DIR / "processed" / "dev_manifest.jsonl"
assert dev_manifest.exists(), f"dev_manifest.jsonl not found at {dev_manifest}! Please check Step 2."
with open(dev_manifest, "r", encoding="utf-8") as f:
    eval_records = [json.loads(line) for line in f][:100]  # evaluate 100 benchmark examples
print(f"Running Zero-Shot evaluation on {len(eval_records)} dev examples...")

correct = 0
valid_sql = 0
for i, r in enumerate(eval_records):
    prompt_text = tokenizer.apply_chat_template(r["prompt"], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False, temperature=None)
    comp = tokenizer.decode(out_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    sql = extract_sql(comp)
    if sql:
        valid_sql += 1
        db_file = DATA_DIR / "raw" / "spider" / "database" / r["db_id"] / f"{r['db_id']}.sqlite"
        ok, rows, trunc, _ = execute_sandbox_query(db_file, sql)
        if ok and not trunc and rows is not None:
            gold_rows = [tuple(x) for x in json.loads(r["gold_result_json"])]
            if Counter(rows) == Counter(gold_rows):
                correct += 1
    if (i + 1) % 25 == 0:
        print(f"  Evaluated {i+1}/{len(eval_records)}: Current EX = {correct/(i+1)*100:.1f}%")

ex_score = correct / len(eval_records) * 100
valid_score = valid_sql / len(eval_records) * 100
print(f"🎯 Zero-Shot Baseline Results: EX = {ex_score:.2f}%, Valid SQL = {valid_score:.2f}%")

# Save baseline evaluation summary so it is included in the final artifact zip
baseline_log_path = OUTPUTS_DIR / "zero_shot_baseline_metrics.json"
with open(baseline_log_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "num_examples": len(eval_records),
            "execution_accuracy_pct": round(ex_score, 2),
            "valid_sql_pct": round(valid_score, 2),
        },
        f,
        indent=2,
    )
print(f"Saved zero-shot metrics to: {baseline_log_path}")

In [ ]:
# Step 6: Configure QLoRA Adapter & SFT Dataset
import json
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# This cell must run exactly once per freshly loaded base model.
if hasattr(model, "peft_config"):
    raise RuntimeError(
        "This model already has a PEFT adapter. Restart the Kaggle session, "
        "rerun Cell 4, and execute Cell 6 only once."
    )

# Disable KV cache during training (required for gradient checkpointing)
model.config.use_cache = False

# Enable non-reentrant gradient checkpointing to save VRAM (~40% reduction) on T4
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
# Required when gradient_checkpointing=True so frozen base layers propagate grads
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)

# Guarantee all trainable LoRA params are FP32 so FP16 GradScaler works safely on T4
for p in model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32)

model.print_trainable_parameters()
print("Trainable adapter dtypes:", sorted({str(p.dtype) for p in model.parameters() if p.requires_grad}))

train_manifest = DATA_DIR / "processed" / "train_manifest.jsonl"
assert train_manifest.exists(), f"train_manifest.jsonl not found at {train_manifest}! Please check Step 2."

train_rows = []
with open(train_manifest, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        full_prompt = tokenizer.apply_chat_template(item["prompt"], tokenize=False, add_generation_prompt=True)
        # Do not manually append eos_token here: TRL 0.25.0 SFTTrainer appends EOS automatically
        full_text = full_prompt + item["completion"]
        train_rows.append({"text": full_text})

train_dataset = Dataset.from_list(train_rows)
print(f"Loaded {len(train_dataset)} training examples.")
assert len(train_dataset) >= 5000, f"Expected ~7,000 training examples, but only found {len(train_dataset)}! Aborting to prevent incomplete training."

In [ ]:
# Step 7: Train QLoRA SFT Baseline on Kaggle GPU (Single-GPU T4-safe settings)
import inspect
import transformers, trl, peft, bitsandbytes

print(
    f"Package versions | transformers={transformers.__version__} | "
    f"trl={trl.__version__} | peft={peft.__version__} | bitsandbytes={bitsandbytes.__version__}"
)

assert torch.cuda.device_count() == 1, (
    f"Detected {torch.cuda.device_count()} visible GPUs. Restart the Kaggle session and run Cell 2 "
    "before importing torch so CUDA_VISIBLE_DEVICES='0' prevents DataParallel wrapping."
)

major, minor = torch.cuda.get_device_capability(0)
# Ampere+ (CC >= 8.0) -> BF16; Turing/T4 (CC 7.5) -> FP16 AMP with FP32 LoRA weights
use_bf16 = (major >= 8)
use_fp16 = not use_bf16
print(
    f"CUDA Capability: {major}.{minor} | Training precision: "
    f"bf16={use_bf16}, fp16={use_fp16} | Optimizer: paged_adamw_8bit | Visible GPUs: {torch.cuda.device_count()}"
)

# Memory-safe training config for T4 (15 GB VRAM)
# Effective batch size = per_device(2) * grad_accum(8) = 16
# If you hit OOM on longer sequences, lower per_device_train_batch_size to 1 and raise gradient_accumulation_steps to 16.
raw_args = {
    "output_dir": str(OUTPUTS_DIR),
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "optim": "paged_adamw_8bit",
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 20,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "logging_steps": 10,
    "save_strategy": "steps",
    "save_steps": 100,
    "save_total_limit": 2,
    "fp16": use_fp16,
    "bf16": use_bf16,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "dataset_text_field": "text",
    "packing": False,
    "report_to": "none",
}

# TRL 0.25.0 uses max_length; keep signature check for version resilience
sig_params = inspect.signature(SFTConfig.__init__).parameters
if "max_length" in sig_params:
    raw_args["max_length"] = 1024
elif "max_seq_length" in sig_params:
    raw_args["max_seq_length"] = 1024

has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig_params.values())
valid_args = raw_args if has_var_keyword else {k: v for k, v in raw_args.items() if k in sig_params}
training_args = SFTConfig(**valid_args)

# Resolve processing_class vs tokenizer across TRL versions
trainer_kwargs = {
    "model": model,
    "train_dataset": train_dataset,
    "args": training_args,
}
trainer_sig = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in trainer_sig:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_sig:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

print("Starting QLoRA SFT Training...")
trainer.train()

# Save trained SFT adapter
sft_adapter_path = OUTPUTS_DIR / "best_adapter"
trainer.model.save_pretrained(str(sft_adapter_path))
tokenizer.save_pretrained(str(sft_adapter_path))
print(f"Saved SFT Adapter to: {sft_adapter_path}")

In [ ]:
# ==========================================================================
# FINAL CELL: MANDATORY ARTIFACT ZIPPER
# Zips the trained SFT adapter and evaluation logs for download
# ==========================================================================
import os
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

output_zip_path = WORKING_DIR / "sql_grpo_sft_artifacts.zip"
print(f"Creating artifact zip at: {output_zip_path} ...")

with zipfile.ZipFile(output_zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(OUTPUTS_DIR):
        for file in files:
            p = Path(root) / file
            zipf.write(p, arcname=str(p.relative_to(WORKING_DIR)))

zip_size_mb = os.path.getsize(output_zip_path) / (1024 * 1024)
print(f"✅ Successfully created: {output_zip_path.name} ({zip_size_mb:.2f} MB)")
print("\n👉 Click the link below to download directly in your browser:")
display(FileLink("sql_grpo_sft_artifacts.zip"))